In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score
from sklearn import model_selection
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn import model_selection
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

In [17]:

dataset = pd.read_csv('Données_V5.csv', sep=';')

print(type(dataset))
print(dataset.head())
print(dataset.info())
print(dataset.shape)
print(dataset.info)

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
<class 'pandas.core.frame.DataFrame'>
              X             Y  OBJECTID created_date     created_user  \
0  1.720320e+06  8.294619e+06         1   02/02/2017  mickael.delaere   
1  1.720898e+06  8.293531e+06         2   02/02/2017  mickael.delaere   
2  1.720894e+06  8.293542e+06         3   02/02/2017  mickael.delaere   
3  1.720902e+06  8.293545e+06         4   02/02/2017  mickael.delaere   
4  1.721089e+06  8.293619e+06         5   02/02/2017  mickael.delaere   

      src_geo              clc_quartier          clc_secteur  haut_tot  \
0  Orthophoto  Quartier du Centre-Ville  Boulevard Richelieu       0.0   
1  Orthophoto  Quartier du Centre-Ville  Boulevard Léon Blum       0.0   
2  Orthophoto  Quartier du Centre-Ville  Boulevard Léon Blum       0.0   
3  Orthophoto  Quartier du Centre-Ville  Boulevard Léon Blum       0.0   
4  Orthophoto  Quartie

In [21]:
haut_col = [dataset.columns[8]]

print(haut_col)
x = dataset[haut_col]

x.head()
print(x)
x = x.dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(x)
print(X_scaled)
max_sil = 0
Ks_test = [2,3,4]
best_k = None

for k in Ks_test:



  kmeans = KMeans(n_clusters=k, random_state=42)
  kmeans.fit(X_scaled)
  klabel = kmeans.fit_predict(X_scaled)

  sil = silhouette_score(X_scaled, klabel)

  print(f"Silhouette score for k={k}: {sil}")

  taille_cluster = pd.Series(klabel).value_counts().sort_index().to_dict()

  print(f"Taille des clusters pour k={k}: {taille_cluster}")


  if max_sil < sil:
    max_sil = sil
    best_k = k
  else :
    max_sil = max_sil
    best_k = best_k

  result = [klabel, sil, taille_cluster]


print("meilleure valeur = ", best_k, "avec pour meilleur score = ", max_sil)


['haut_tot']
       haut_tot
0           0.0
1           0.0
2           0.0
3           0.0
4           0.0
...         ...
11243       NaN
11244       3.0
11245       4.0
11246       3.0
11247       3.0

[11248 rows x 1 columns]
[[-1.78593899]
 [-1.78593899]
 [-1.78593899]
 ...
 [-1.09500801]
 [-1.26774075]
 [-1.26774075]]
Silhouette score for k=2: 0.6539451814099958
Taille des clusters pour k=2: {0: 3949, 1: 7244}
Silhouette score for k=3: 0.6201936090886526
Taille des clusters pour k=3: {0: 3776, 1: 6292, 2: 1125}
Silhouette score for k=4: 0.5371046350395285
Taille des clusters pour k=4: {0: 2824, 1: 3156, 2: 1125, 3: 4088}
meilleure valeur =  2 avec pour meilleur score =  0.6539451814099958


In [22]:
final_kmeans2 = KMeans(n_clusters=2, random_state=42)
final_klabel2 = final_kmeans2.fit_predict(X_scaled)

cluster_series_k2 = pd.Series(final_klabel2, index=x.index)
dataset['cluster_taille'] = cluster_series_k2

cluster_mapping_k2 = {0: 'grand', 1: 'petit'}
dataset['cluster_taille2'] = dataset['cluster_taille'].map(cluster_mapping_k2)

final_kmeans3 = KMeans(n_clusters=3, random_state=42)
final_klabel3 = final_kmeans3.fit_predict(X_scaled)

cluster_series_k3 = pd.Series(final_klabel3, index=x.index)
dataset['cluster_taille3'] = cluster_series_k3

cluster_mapping_k3 = {0: 'moyen', 1: 'petit', 2: 'grand'}
dataset['cluster_taille3'] = dataset['cluster_taille3'].map(cluster_mapping_k3)

print("DataFrame avec les nouvelles colonnes de cluster:")
print(dataset[['haut_tot', 'cluster_taille', 'cluster_taille2', 'cluster_taille3']].head(10))


DataFrame avec les nouvelles colonnes de cluster:
   haut_tot  cluster_taille cluster_taille2 cluster_taille3
0       0.0             1.0           petit           petit
1       0.0             1.0           petit           petit
2       0.0             1.0           petit           petit
3       0.0             1.0           petit           petit
4       0.0             1.0           petit           petit
5       0.0             1.0           petit           petit
6       0.0             1.0           petit           petit
7       0.0             1.0           petit           petit
8       0.0             1.0           petit           petit
9       6.0             1.0           petit           petit


In [20]:
dataset.to_csv("dataset_ajuste.csv", index=False)